# Tool use: reminder agent

Give Claude three **user-implemented** tools (current datetime, add duration to datetime, set reminder) and let it chain them to answer a user request.
"User-implemented" means you write both the JSON schema and the Python handler.
(Anthropic also ships built-in tools where part or all of the implementation lives on Anthropic's side; those come later in the chapter.)

## Flow

`run_conversation` is a loop:

1. Send messages + tool schemas to Claude.
2. If the response has `stop_reason="tool_use"`, execute every `tool_use` block in `run_tools`, append the `tool_result` blocks back as a user message, and loop.
3. Otherwise, break and return the final messages.

Model: `claude-haiku-4-5` (default from `chat_tools`).
## Generating a JSON schema for a tool

In a new Claude conversation, run this prompt (replace `<paste the function code here>` with the actual function source):

```
Write a valid JSON schema for tool calling for this function:
<paste the function code here>

Follow the best practices in the Anthropic tool use documentation:
https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview.
```


In [7]:
import json
from datetime import datetime, timedelta

from chat_tools import add_assistant_message, add_user_message, chat, text_from_message
from tool_schemas import (
    add_duration_to_datetime_schema,
    get_current_datetime_schema,
    set_reminder_schema,
)

In [8]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        days_in_month = [
            31,
            29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
            31,
            30,
            31,
            30,
            31,
            31,
            30,
            31,
            30,
            31,
        ]
        day = min(date.day, days_in_month[month - 1])
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")
    return f"Reminder set for {timestamp}"

In [10]:
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }
        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [11]:
def run_conversation(messages):
    tools = [get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema]
    while True:
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            print("---\n")
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [12]:
messages = []
add_user_message(
    messages,
    "Set a reminder for my doctor's appointment. It's 177 days after Jan 1st, 2050.",
)
run_conversation(messages)


Now I'll set a reminder for your doctor's appointment on June 27, 2050:
----
Setting the following reminder for 2050-06-27T00:00:00:
Doctor's appointment
----
Done! I've set a reminder for your doctor's appointment on **Monday, June 27, 2050** at midnight. The date is 177 days after January 1st, 2050.
---



[{'role': 'user',
  'content': "Set a reminder for my doctor's appointment. It's 177 days after Jan 1st, 2050."},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_017xjRqEt6x1dK2hJXrrBkZ7', caller=DirectCaller(type='direct'), input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days'}, name='add_duration_to_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_017xjRqEt6x1dK2hJXrrBkZ7',
    'content': '"Monday, June 27, 2050 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set a reminder for your doctor's appointment on June 27, 2050:", type='text'),
   ToolUseBlock(id='toolu_01LAKDueNAW4yjTUa2w4BhjJ', caller=DirectCaller(type='direct'), input={'content': "Doctor's appointment", 'timestamp': '2050-06-27T00:00:00'}, name='set_reminder', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool